# Macrophage phosphoproteomics analysis


In [ ]:
import polars as pl
import json
import polars.selectors as cs
from sklearn.decomposition import PCA
import scipy
from pathlib import Path
import pandas as pd
import numpy as np
import math
from functools import reduce
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from plotnine import *
from Bio import SeqIO
from src.uniprot_utils import create_entry_cache, get_function
import os
from collections import defaultdict
import scipy.stats as stat


WP_PATH = Path('/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/01_Data Analysis/01_Proteomics/01_unenriched proteomics/03_results/20250219_3reps/03_combined_files/1/01_combcond_percctrl_wp_keratins_removed_min4rep_filter.csv')
treatment_condition = "TLR4"
control_condition = "M0"
cols = {"M0": "#AAAAAA", "TLR4": "#FFCC31"} # color palette
CONDITIONS = [treatment_condition, control_condition]
# pipeline output from a branch allowing multiple mods
raw_data_dir = Path("/Users/henrysanford/dev/test_data/phospho/phospho_no_step2_agg") 


# **Preprocessing**

## Pull data for all donors

Collect phosphoproteomics data from different donors and searches, attatch sample names, and format into one tidy dataframe

In [ ]:
# Collect all processed dfs
dfs = []
suffix = "2/01_processed_files"
for dir in os.listdir(raw_data_dir):
    if "output" not in dir:
        continue
    n_diffs = str(dir).split("_")[1].split("diff")[0]
    for file in os.listdir(raw_data_dir / dir / suffix):
        if "processed" in file:
            print(f"Reading {file}, {n_diffs} diff mods")
            experiment_name = file.split("census-out_")[1].split(".csv")[0]
            df = pl.read_csv(raw_data_dir / dir / suffix / file).with_columns(
                pl.lit(n_diffs).alias("n_diff_mods"),
                pl.lit(experiment_name).alias("experiment"),
            )
            dfs.append(df)
df = pl.concat(dfs, how="vertical_relaxed")

# Add channel names
metadata = (
    pl.read_csv(raw_data_dir / "Macrophage phosphoproteomics experiments - Sheet1.csv")
    .drop(cs.contains("peptide ids"), "abbreviation")
    .unpivot(
        on=cs.contains("tmt_channel_"),
        index=~cs.contains("tmt_channel_"),
        value_name="channel_name",
    )
    .with_columns(pl.col("experiment").str.replace("-", "_", literal=True))
)
tag_columns = [col for col in df.columns if col.startswith("tag_")]
rename_mapping = {col: f"tmt_channel_{i}" for i, col in enumerate(tag_columns, 1)}

clean_df = (
    df.rename(mapping=rename_mapping)
    .with_columns(pl.col("experiment").str.replace("-", "_", literal=True))
    .unpivot(
        on=cs.contains("tmt_channel_"),
        index=~cs.contains("tmt_channel_"),
        value_name="channel_ratio",
    )
    .join(other=metadata, on=["experiment", "variable"], how="left")
).drop(["experiment", "variable"])


def unnest_channel_name(df):
    return df.with_columns(
        pl.col("channel_name")
        .str.splitn("_", 3)
        .struct.rename_fields(["cell_state", "donor", "technical_replicate"]),
    ).unnest("channel_name")


peptide_results_clean = unnest_channel_name(
    clean_df.select(
        [
            "uniprot",
            "residue",
            "channel_name",
            "channel_ratio",
            "description",
            "sequence",
             "n_diff_mods",
        ]
    )
)


peptide_results_clean = (
    peptide_results_clean.filter(~pl.col("uniprot").str.contains("contaminant"))
    .with_columns(
        # we are not removing mods from the sequence
        # so sequence aggregation won't average multiple mods
        pl.col("sequence").str.split(".").list.get(1),#.str.replace_all("\\*", ""),
        pl.col("sequence")
        .str.count_matches(pattern="*", literal=True)
        .alias("num_modifications"),
        pl.when(pl.col("description").str.contains("GN="))
        .then(pl.col("description").str.extract(r"GN=(\S+)"))
        .otherwise(pl.col("uniprot"))
        .alias("protein"),
    )
    .sort(by="num_modifications")
)

peptide_results_clean

sns.histplot(
    data=peptide_results_clean.select(pl.col("sequence"), pl.col("num_modifications")),
    y="num_modifications",binwidth=1
)

## Aggregating S, T, and Y across donors
- Step 1: aggregate multiple peptides containing the same peptide
- Step 2: handle peptide fully contained in another peptide

In [ ]:
all_id_cols = [
    "uniprot",
    "protein",
    "cell_state",
    "donor",
    "technical_replicate",
    "description",
]

# Separate donor from other ID columns for cross-donor aggregation
base_id_cols = [col for col in all_id_cols if col != "donor"]


print(
    f"Starting peptides: {len(peptide_results_clean.select(["uniprot","sequence"]).unique())}"
)
# aggregate residues so multiple peptides containing the same residue are averaged


def get_shortest_string(sequences, aas_to_aggregate):
    """
    Find shortest common string in list containing amino acids to aggregate
    """
    # polars series to python list
    sequences = list(sequences)

    # handle empty or single item cases
    if not sequences:
        return None
    if len(sequences) == 1:
        return sequences[0]

    # get sequences with minimum length
    min_len = min(len(x) for x in sequences if x is not None)
    sequence_list = [x for x in sequences if x is not None and len(x) == min_len]

    if len(sequence_list) == 1:
        return sequence_list[0]

    if len(set(sequence_list)) == 1:
        return sequence_list[0]  # all sequences are equal

    # find sequence with amino acid closest to N-terminus
    tmp_list = []
    seq_indices = []

    for i, seq in enumerate(sequence_list):
        for aa in aas_to_aggregate:
            finder = seq.find(aa)
            tmp_list.append(finder)
            seq_indices.append(i)

    if tmp_list:
        tmp_list = [float("inf") if x == -1 else x for x in tmp_list]
        min_index = tmp_list.index(min(tmp_list))
        sequence = sequence_list[seq_indices[min_index]]
    else:
        sequence = sequence_list[0]

    return sequence


def average_seqs(original_df, aggregated_seqs_df):
    """
    join with aggregation mappings and average peptides that have been aggregated
    take average (mean) for aggregated residues
    """
    return (
        original_df.join(aggregated_seqs_df, on=["uniprot", "sequence"], how="left")
        .with_columns(
            pl.col("sequence_representative").fill_null(
                pl.col("sequence")
            )  # fallback to original if no mapping found
        )
        .drop("sequence")
        .rename({"sequence_representative": "sequence"})
        .group_by(all_id_cols + ["sequence"])
        .agg(
            pl.col("channel_ratio").mean(),
            pl.col("residue").unique().sort().str.join(", "),
        )
    )


# build mapping keyed on (uniprot, sequence, num_modifications)
shortest_seq_rows = []
for uniprot, residue, num_modifications, seqs in (
    peptide_results_clean.group_by("uniprot", "residue", "num_modifications")
    .agg(pl.col("sequence").unique())
    .rows()
):
    for seq in seqs:
        shortest_seq_rows.append(
            {
                "uniprot": uniprot,
                "sequence": seq,
                "sequence_representative": get_shortest_string(seqs, ["S", "T", "Y"]),
            }
        )

shortest_seqs_df = pl.DataFrame(shortest_seq_rows).unique(
    subset=["uniprot", "sequence"]
)  # guard against any duplicates


sequence_means = average_seqs(
    original_df=peptide_results_clean, aggregated_seqs_df=shortest_seqs_df
)

print(f"after step 1: {len(sequence_means.select(["uniprot","sequence"]).unique())}")

In [ ]:
# handle peptide that is fully contained in another peptide
perform_step_2 = True
def solveContainedSequences(sequence_string, list_of_unique_seq, aa_to_aggregate):
    # Find all sequences that contain or are contained by sequence_string
    contained = [
        item
        for item in list_of_unique_seq
        if sequence_string in item or item in sequence_string
    ]

    if len(contained) <= 1:
        return sequence_string

    # Get shortest sequence (parent)
    parent_string = min(contained, key=len)

    # Check if leftover contains target amino acids
    leftover = sequence_string.replace(parent_string, "")
    aa_set = set(aa_to_aggregate)

    return sequence_string if aa_set & set(leftover) else parent_string

if(perform_step_2):
    # Create aggregation mapping across all donors for each uniprot
    aggregated_seq_rows = []
    for uniprot, seqs in (
        sequence_means.group_by("uniprot").agg(pl.col("sequence").unique()).rows()
    ):
        for seq in seqs:
            aggregated_seq_rows.append({
                "uniprot": uniprot,
                "sequence": seq,
                "sequence_representative": solveContainedSequences(seq, seqs, ["S","T","Y"]),
            })

    aggregated_seqs_df = (
        pl.DataFrame(aggregated_seq_rows)
        .unique(subset=["uniprot", "sequence"])
    )

    # Apply the sequence aggregation mapping and then re-aggregate by donor
    sequence_means = average_seqs(original_df=sequence_means, aggregated_seqs_df=aggregated_seqs_df)

    print(f"after step 2: {len(sequence_means.select(["uniprot","sequence"]).unique())}")

sequence_to_residue = defaultdict(set)
for row in (
    sequence_means.select(["sequence", "residue"]).unique().iter_rows(named=True)
):
    # Split on "," and strip whitespace
    residues = row["residue"].replace(";", ",").split(",")
    sequence_to_residue[row["sequence"]].update(r.strip() for r in residues)
clean_sequence_to_residue = {}
for sequence in sequence_to_residue.keys():
    clean_sequence_to_residue[sequence] = ",".join(sequence_to_residue[sequence])

# reformat to residue x channel matrix
df = (
    (
        sequence_means.with_columns(
            pl.concat_str(
                ["cell_state", "donor", "technical_replicate"], separator="_"
            ).alias("channel_name")
        )
        .drop(["cell_state", "donor", "technical_replicate"])
        .pivot(
            on="channel_name",
            values="channel_ratio",
            index=["uniprot", "protein", "description", "sequence"],
        )
    )
    .with_columns(
        pl.col("sequence").replace(clean_sequence_to_residue).alias("residue")
    )
    .with_columns(
        pl.concat_str(
            ["uniprot", "protein", "description", "residue", "sequence"], separator="_"
        ).alias("identifier")
    )
)

## Median normalization

Normalize treatment condition samples to each other by calculating normalization factors

In [ ]:
def channel_ratio_boxplot(df, ax):
    data = (
        df.drop(
            [
                "uniprot",
                "sequence",
                "residue",
                "description",
                "protein",
            ]
        )
        .unpivot(
            index="identifier", value_name="Channel ratio", variable_name="channel"
        )
        .with_columns(pl.col("channel").str.split("_process").list.get(0))
    )
    sns.boxplot(
        data=data, y="channel", x="Channel ratio", log_scale=False, showfliers=None, ax=ax, 
    )


fig, axs = plt.subplots(ncols=2, sharey=True, sharex = True)
axs[0].set_title("before normalization")
axs[1].set_title("after normalization")

channel_ratio_boxplot(df.select(sorted(df.columns)), axs[0])


def normalization_treatment_condition(df, treatment_labelling, index_cols):
    df = df.to_pandas().set_index(index_cols)
    norm_factor_series = []

    cond_columns = df.filter(like=treatment_labelling).columns.tolist()
    sub_df = df[cond_columns]

    norm_factors = sub_df.median().median() / sub_df.median()
    norm_factor_series.append(norm_factors)
    norm_factor = pd.concat(norm_factor_series, axis=1).sum(1)
    normalized_df = df[norm_factor.index] * norm_factor
    list_of_ctrl_cols = [
        column for column in df.columns if column not in norm_factor.index
    ]
    untouched_ctrl_columns = df[list_of_ctrl_cols]
    norm_df = normalized_df.reset_index().merge(
        untouched_ctrl_columns.reset_index(), on=index_cols
    )
    norm_df = norm_df[df.reset_index().columns]
    return pl.from_pandas(norm_df)


id_cols = [
    "uniprot",
    "sequence",
    "residue",
    "description",
    "protein",
    "identifier"
]
normalized_df = df.clone()
for condition in CONDITIONS:
    normalized_df = normalization_treatment_condition(
    df=normalized_df, treatment_labelling=condition, index_cols=id_cols
    )

channel_ratio_boxplot(normalized_df.select(sorted(normalized_df.columns)), axs[1])

# Principal component analysis

In [ ]:
def get_pca_plot(df, index_cols, title_string, out_dir):
    """Principal component analysis.
    Scales data with log. Runs and plots PCA with scikit learn
    """
    out_dir.mkdir(exist_ok=True)
    # drop na
    df = df.dropna()
    df = df.set_index(index_cols)
    # log2 transform
    df_log = np.log2(df)

    df_log = df_log.replace(-np.inf, np.nan)
    df_log = df_log.dropna()

    number_of_proteins_in_common = df_log.shape[0]
    print(number_of_proteins_in_common)

    # transpose table
    df_log_t = df_log.transpose()
    df_log_t.reset_index(inplace=True)
    df_log_t = df_log_t.rename(columns={"index": "channel_name"})

    # get X and y
    X = df_log_t.drop("channel_name", axis=1)

    # get PCA 2, fit transform, get df
    pca = PCA(n_components=3)
    principalComponents = pca.fit_transform(X)
    principalDf = pd.DataFrame(
        data=principalComponents,
        columns=[
            "principal component 1",
            "principal component 2",
            "principal component 3",
        ],
    )
    # add channel name
    finalDf = pd.concat([principalDf, df_log_t[["channel_name"]]], axis=1)

    # get PCA %
    pca_1_percent = round((pca.explained_variance_ratio_[0] * 100), 2)
    pca_2_percent = round((pca.explained_variance_ratio_[1] * 100), 2)
    pca_3_percent = round((pca.explained_variance_ratio_[2] * 100), 2)

    percent_df = pd.DataFrame(
        [
            {"principal_component": "PC1", "percent_explained": pca_1_percent},
            {"principal_component": "PC2", "percent_explained": pca_2_percent},
            {"principal_component": "PC3", "percent_explained": pca_3_percent},
        ]
    )

    # add condition
    finalDf["condition"] = finalDf["channel_name"]

    finalDf["condition"] = finalDf["condition"].str.split("_").str[0]
    color_discrete_sequence_list = [
        "salmon",
        "#A3A533",
        "#56BC82",
        "#4EADF0",
        "#D772EC",
    ]
    title_info = title_string + " (" + str(number_of_proteins_in_common) + " Proteins)"

    fig = px.scatter(
        finalDf,
        x="principal component 1",
        y="principal component 2",
        hover_data=["channel_name"],
        color=finalDf["condition"],
        color_discrete_sequence=color_discrete_sequence_list,
        labels={
            "principal component 1": "PC1 ({}%)".format(pca_1_percent),
            "principal component 2": "PC2 ({}%)".format(pca_2_percent),
            "condition": "Condition",
        },
        title=title_info,
        template="plotly_white",
    )

    fig.update_traces(marker=dict(size=12), selector=dict(mode="markers"))
    fig.update_layout(height=600, width=600, showlegend=True, legend_title_text="")

    loadings_df = pd.DataFrame(
        data=np.transpose(pca.components_), columns=["PC1", "PC2", "PC3"]
    )
    loadings_df["variable"] = df_log.index.tolist()

    loadings_df = loadings_df.set_index("variable")

    finalDf.to_csv(out_dir / "pca_results.csv")
    loadings_df.to_csv(out_dir / "loadings_results.csv")
    percent_df.to_csv(out_dir / "percent_explained.csv")

    return (fig, loadings_df, finalDf, percent_df)


def save_plot(p, fn, width, height):
    p.save(f"{fn}.pdf", height=height, width=width, format = "pdf")


POINT_STROKE = 0.25
POINT_SIZE = 2.5
LINE_WIDTH = 0.125
FONT_SIZE = 6
FC_CUTOFF = np.log2(1.5)


def my_theme():
    return theme_classic() + theme(
        axis_text=element_text(size=FONT_SIZE, color="black", family="Arial"),
        axis_title=element_text(size=FONT_SIZE, color="black", family="Arial"),
        legend_title=element_text(size=FONT_SIZE, color="black", family="Arial"),
        legend_text=element_text(size=6, color="black", family="Arial"),
        axis_line=element_blank(),
        axis_ticks=element_line(color="black", size=LINE_WIDTH),
        axis_ticks_minor=element_blank(),
        panel_border=element_rect(color="black", fill=None, size=LINE_WIDTH * 2),
        plot_title=element_text(size=6, color="black", family="Arial", ha="center"),
        aspect_ratio=1,
        figure_size=(4, 2),
    )


def plot_loadings(loadings_df, ggplot_obj, arrow_scaling, num_loadings=5):
    top_pc1 = loadings_df.sort("PC1").tail(num_loadings).select("variable").to_series()
    bottom_pc1 = (
        loadings_df.sort("PC1").head(num_loadings).select("variable").to_series()
    )
    top_pc2 = loadings_df.sort("PC2").tail(num_loadings).select("variable").to_series()
    bottom_pc2 = (
        loadings_df.sort("PC2").head(num_loadings).select("variable").to_series()
    )

    filtered_loadings = loadings_df.filter(
        pl.col("variable").is_in(
            list(top_pc1) + list(bottom_pc1) + list(top_pc2) + list(bottom_pc2)
        )
    )

    return (
        ggplot_obj
        + geom_segment(
            filtered_loadings,
            aes(x=0, y=0, xend="PC1*arrow_scaling", yend="PC2*arrow_scaling"),
            arrow=arrow(length=0.1),
            size=LINE_WIDTH,
            color="grey",
            inherit_aes=False,
        )
        + geom_text(
            filtered_loadings,
            aes(x="PC1*arrow_scaling", y="PC2*arrow_scaling", label="variable"),
            size=6,
            color="black",
            inherit_aes=False,
        )
    )


def pca_plot(df, loadings_df, x_column, y_column, x_lab, y_lab, fn, arrow_scaling=100):
    plt = (
        ggplot(df, aes(x=x_column, y=y_column, fill="condition"))
        + geom_point(
            size=POINT_SIZE, stroke=POINT_STROKE, shape="o", alpha=0.9, color="black"
        )
        + my_theme()
        + labs(x=x_lab, y=y_lab)
        + scale_fill_manual(values=cols, name="")
    )

    plt.show()

    save_plot(plt, fn, height=2, width=4)

    plot_with_loadings = plot_loadings(loadings_df, plt, arrow_scaling)
    plot_with_loadings.show()
    save_plot(plot_with_loadings, f"{fn}_with_loadings", height=2, width=4)

    base_plot = ggplot() + my_theme() + labs(x=x_lab, y=y_lab)
    loadings_only = plot_loadings(loadings_df, base_plot, 1, 10)
    loadings_only.show()
    save_plot(loadings_only, f"{fn}_loadings_only", height=2, width=4)


def run_and_plot_pca(df):
    pca_plt, loadings_df, pca_df, percent_explained_df = get_pca_plot(
        df = df,
        index_cols = "ID",
        title_string="Phosphoproteomics",
        out_dir=Path("./")
    )
    pca_plt.show()


    pca_dir = "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Vinogradova Laboratory/Henry_data processing/01_data_analysis_folders/06_macrophage/macrophage/phosphoproteomics/"

    loadings_df = pl.from_pandas(loadings_df.reset_index())
    
    pca_plot(
        pl.from_pandas(pca_df),
        loadings_df,
        "principal component 1",
        "principal component 2",
        f"PC1 ({percent_explained_df.to_dict(orient = "records")[0]["percent_explained"]} %)",
        f"PC2 ({percent_explained_df.to_dict(orient = "records")[1]["percent_explained"]} %)",
        f"{pca_dir}pca",
        arrow_scaling=260,
    )
pca_input_df = normalized_df
not_normalized_df = pca_input_df.with_columns(
    pl.concat_str(
        pl.col("protein"), pl.lit("_"), pl.col("residue")
    ).str.replace(pattern = "; ", value = ",").alias("ID")
).drop(
    ["identifier", "residue", "protein", "uniprot", "description",  "sequence"]
).to_pandas()

run_and_plot_pca(not_normalized_df)

# Normalize replicate level data to WP

Normalize by dividing the channel ratio by the whole proteome FC. This way the replicate level data is normalized, not just the averaged FC. Not every protein from the phospho data had WP expression data, so the normalized data has 9,591 phosphopeptides, compared with 11,690 in the un-normalized output

In [ ]:
pca_input_df.select(["uniprot", "sequence", "residue"]).unique().shape
pl.from_pandas(normalized_to_wp).select(["ID"]).unique().shape

In [ ]:
wp_df = (
    pl.read_csv(WP_PATH)
    .drop(["", "protein", "description"])
    .unpivot(index=["uniprot"], variable_name="condition")
    .with_columns(pl.col("value").truediv(100).alias("FC"))
)

normalized_to_wp = (
    pca_input_df.with_columns(
        pl.concat_str(pl.col("protein"), pl.lit("_"), pl.col("residue"))
        .str.replace(pattern="; ", value=",")
        .alias("ID")
    )
    .drop(["sequence", "residue", "description", "protein", "identifier"])
    .unpivot(index=["uniprot", "ID"])
    .with_columns(pl.col("variable").str.split("_").list.get(0).alias("condition"))
    .join(other=wp_df, on=["uniprot", "condition"], how="inner")
    .with_columns(
        pl.col("value").truediv(pl.col("FC")).alias("normalized_channel_ratio")
    )
    .select(["ID", "variable", "normalized_channel_ratio"])
    .unique(subset=["ID", "variable"])
    .pivot(on="variable", values="normalized_channel_ratio")
    .to_pandas()
)


run_and_plot_pca(normalized_to_wp)

Compare expression normalized and not expression normalized data

In [ ]:
df = pl.concat(
    [
        pl.from_pandas(not_normalized_df)
        .unpivot(index="ID")
        .with_columns(pl.lit("not normalized").alias("normalization")),
        pl.from_pandas(normalized_to_wp)
        .unpivot(index="ID")
        .with_columns(pl.lit("normalized to WP").alias("normalization")),
    ]
)

sns.boxplot(
    data=df.sort(by = "variable"),
    y="variable",
    x="value",
    hue = "normalization",
    log_scale=True,
    showfliers = False
)

# Volcano plot

In [ ]:

def get_expr(row, fc_cutoff = 2):
    p_value_column = '-log10_pval'
    p_value_cutoff = -1*math.log10(0.05)
    raw_fc_cutoff = fc_cutoff
    fc_cutoff = math.log2(raw_fc_cutoff)
    if (row['log2_FC'] > fc_cutoff) & (row[p_value_column] > p_value_cutoff):
        return "Significant Up"
    if (row['log2_FC'] > fc_cutoff) & (row[p_value_column] < p_value_cutoff):
        return "Not Significant Up"
    if (row['log2_FC'] < -fc_cutoff) & (row[p_value_column] > p_value_cutoff):
        return "Significant Down"
    if (row['log2_FC'] < -fc_cutoff) & (row[p_value_column] < p_value_cutoff):
        return "Not Significant Down"
    if (row['log2_FC'] > -fc_cutoff) & (row['log2_FC'] < fc_cutoff) & (row[p_value_column] > p_value_cutoff):
        return "Significant but <{} FC".format(str(raw_fc_cutoff))
    else:
        return "Not Significant"

def get_p_value(row, cond_1, cond_2):
    ttest_result = stat.ttest_ind(row[cond_1],  row[cond_2], nan_policy='omit')
    return ttest_result[1]

def get_volcano_plot_treatment_vs_control(conditions_list, control_labelling, df, file_name, folder_path, index_cols = [], data_type = "Residues", fc_cutoff = 2):
    volcano_df_list = []

    for condition in conditions_list:
        copy_df = df.copy()
        list_cond_1 = copy_df.filter(like=condition).columns.tolist()
        list_cond_2 = copy_df.filter(like=control_labelling).columns.tolist()
        if len(list_cond_1) <= 1 or len(list_cond_2) <= 1:
            print(f"Condition {condition} does not have enough replicates to be shown in volcano plot!")
            print(list_cond_1)
            print(list_cond_2)
            continue

        for labelling in [control_labelling, condition]:
            list_columns_labelling = copy_df.filter(like=labelling).columns.tolist()
            labelling_df = copy_df[list_columns_labelling]
            copy_df["mean_"+labelling] = labelling_df.mean(axis=1)
            
        copy_df["FC"] = copy_df["mean_"+condition] / copy_df["mean_"+control_labelling]

        idx_cond_1 = copy_df.columns.get_indexer(list_cond_1)
        idx_cond_2 = copy_df.columns.get_indexer(list_cond_2)
        copy_df["p_value"] = copy_df.apply(get_p_value, axis=1, args=(idx_cond_1, idx_cond_2), )
        
        copy_df["log2_FC"] = np.log2(copy_df["FC"])
        volcano_df = copy_df[["p_value", "log2_FC", "FC"]]
            
        volcano_df = volcano_df.dropna() 
        volcano_df["-log10_pval"] = -1*np.log10(volcano_df["p_value"])
        volcano_df["Regulation"] = volcano_df.apply(get_expr, axis=1, fc_cutoff=fc_cutoff)
            
        volcano_df["Regulation"] = volcano_df["Regulation"].astype('category')
        
        volcano_df = volcano_df.reset_index()

        title_name = file_name + " - " + condition + " vs. " + control_labelling + " (" + str(len(volcano_df)) + " " + data_type  + ")"

        volcano_df = volcano_df.reset_index().set_index(index_cols)
        volcano_df = volcano_df.add_suffix("_"+title_name)
        volcano_df_list.append(volcano_df)
            
    return pd.concat(volcano_df_list, axis=1)



def convert_to_long_format(df, id_vars, values_name):
    df = (
        # convert to long format
        df.unpivot(
            index=id_vars,
            value_name=values_name,
            variable_name="sample",
        ).drop_nans(subset=values_name)
        .drop_nulls(subset=values_name)
        # extract donor from sample name
        .with_columns(
            pl.col("sample").str.split("_").list.get(1).alias("donor"),
            pl.col("sample").str.split("_").list.get(0).alias("condition"),
        )
    )

    return df

def filter_to_min_2_reps(
    df: pl.DataFrame, id_vars: list, data_prefix: str
) -> pl.DataFrame:
    """Filters dataframe to targets quantified in
    2 biological replicates"""
    values_name = f"{data_prefix}"
    df = convert_to_long_format(df, id_vars, values_name)
    # filter to min two biological replicates
    return (
        df.group_by(id_vars)
        .agg(pl.col("donor").n_unique().alias("n_replicates"))
        .filter(pl.col("n_replicates") > 1)
        .drop("n_replicates")
        .join(other=df, on=id_vars)
    )

filtered = filter_to_min_2_reps(
    df = pl.from_pandas(normalized_to_wp),
    id_vars = ["ID"],
    data_prefix="cr"
)

volcano_data = get_volcano_plot_treatment_vs_control(
    conditions_list=[treatment_condition],
    control_labelling=control_condition,
    df=filtered.drop("condition", "donor").pivot(on = "sample", values="cr").to_pandas().set_index("ID"),
    file_name="phospho",
    folder_path="/Users/henrysanford/dev/test_data/phospho/tiff_3reps/output/8/05_results/volcano_plot",
    index_cols=["ID"]
).reset_index()

volcano_data["ID"] = volcano_data["ID"].str.replace(";", ",").str.replace(" ","")


In [ ]:
# build protein metadata to attatch to quant
df = normalized_df.select(id_cols).unique().filter(
    pl.col("uniprot").str.contains("contaminant").not_(),
    pl.col("description").str.contains("Keratin").not_(),
)
cache = create_entry_cache(df)
formatted_df = df.with_columns(
    pl.col("uniprot")
    .map_elements(lambda x: get_function(x, cache))
    .alias("uniprot_function")
)

# join metadata with quant results
formatted_df = (
    pl.from_pandas(volcano_data)
    .with_columns(
        pl.col("ID")
        .str.split_exact("_", 1)
        .struct.rename_fields(["protein", "residue"]),
        pl.col("ID").alias("ID_1"),
    )
    .unnest("ID")
    .join(other=formatted_df, on=["protein", "residue"], how="left")
)

# join replicate level data with t test and metadata results
replicate_level = filtered.drop("condition", "donor").pivot(on="sample", values="cr")
replicate_level = replicate_level.select(sorted(replicate_level.columns))


def sort_residues(residues):
    if "," not in residues:
        return residues
    residues = residues.split(",")
    residue_indices = [int(residue[1:]) for residue in residues]
    return ",".join([x for _, x in sorted(zip(residue_indices, residues))])

def sort_residue_column(df):
    unsorted_residues = set(df["residue"])
    residue_dict = {}
    for residues in list(unsorted_residues):
        residue_dict[residues] = sort_residues(residues)
    return df.with_columns(pl.col("residue").replace(residue_dict))
formatted_df = sort_residue_column(formatted_df)


formatted_df.rename({"ID_1": "ID"}).join(
    other=replicate_level, on="ID", how="left"
).write_csv("normalized_phosphorylation_table.csv")

# Structural features 


In [ ]:
white_2023_pse = pl.read_csv(
    "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Railia_Mass spectrometry/Mass-spectrometry/6_in_silico_digestion/reference_data/20230315_Supplementary_AlphaFold_pPSE.csv",
    ignore_errors=True,
).rename({"protein_id": "uniprot", "position": "pos"})

n_residues = formatted_df.shape[0]
formatted_df.with_columns(
    pl.col("residue")
    .str.split(",")
    .list.get(0)
    .str.slice(1)
    .cast(pl.Int64)
    .alias("pos")
).join(other=white_2023_pse, on=["pos", "uniprot"]).filter(
    pl.col(f"p_value_phospho - TLR4 vs. M0 ({n_residues} Residues)").lt(0.05),
    pl.col(f"log2_FC_phospho - TLR4 vs. M0 ({n_residues} Residues)").gt(2),
).write_csv(
    "structural_features.csv"
)

# Motif enrichment analysis
Prep data for motif analysis

In [ ]:
def read_fasta(fn):
    seqs = {}
    for record in SeqIO.parse(fn, format="fasta"):
        uniprot_id = str(record.id).split("|")[1]
        seqs[uniprot_id] = str(record.seq).replace("*", "")
    return seqs


def add_integer_location(df):
    df = (
        df.with_columns(pl.col("residue").str.replace(" ", "").str.split(","))
        .explode(pl.col("residue"))
        .with_columns(
            pl.col("residue")
            .str.replace(" ", "")
            .str.slice(offset=1)
            .cast(pl.Int64)
            .sub(1)
            .alias("modification_locations"),
            pl.col("residue").str.head(1).alias("amino_acid"),
        )
    )
    return df

def fetch_flanking_seq(r):
    flank_len = 5
    protein_sequence = r["uniprot_sequence"]
    protein_len = len(protein_sequence)

    cys_index = r["modification_locations"] - 1

    start = cys_index - flank_len
    start = max(0, start)

    stop = cys_index + flank_len + 1
    stop = min(stop, protein_len)

    r["flanking_seq"] = protein_sequence[start:stop]
    return r

In [ ]:
flank_len = 7
uniprot = read_fasta(
    "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Vinogradova Laboratory/Henry_data processing/03_fasta_files/homo_sapiens/20240312/uniprotkb_proteome_UP000005640_AND_revi_2024_03_12.fasta"
)

p_val = f"p_value_phospho - TLR4 vs. M0 ({n_residues} Residues)"
lfc = f"log2_FC_phospho - TLR4 vs. M0 ({n_residues} Residues)"


kl_df = formatted_df.with_columns(
    pl.col("uniprot").replace(uniprot).alias("uniprot_sequence")
)
kl_df = add_integer_location(kl_df)
kl_df  = kl_df.select(
    p_val,
    lfc,
    "modification_locations",
    "uniprot_sequence",
    "protein",
    "residue",
    "amino_acid"
).with_columns(
    pl.col("uniprot_sequence").str.len_chars().alias("protein_len")
).with_columns(
    pl.col("modification_locations").sub(flank_len).clip(lower_bound=0).alias("start"),
    (pl.col("modification_locations") + flank_len + 1)
    .clip(upper_bound=pl.col("protein_len"))
    .alias("stop"),
).with_columns(
    [
        pl.col("uniprot_sequence")
        .str.slice(pl.col("start"), (pl.col("stop") - pl.col("start")))
        .alias("flanking_seq")
    ]
).with_columns(
    pl.col("flanking_seq").str.len_chars().alias("flanking_seq_len"),
)


kl_df.filter(
    pl.col("flanking_seq_len") == 15
).select(
    "protein",
    "residue",
    "flanking_seq",
    p_val,
    lfc,
    "amino_acid"
).filter(
    pl.col(lfc).is_finite()
).write_csv(
    "/Users/henrysanford/dev/phosphorylaiton_motifs.tsv", separator="\t"
)

# Cross reference with reactivity and phosphositeplus data

To download PhosphositePlus

```sh
wget http://www.phosphosite.org/downloads/Phosphorylation_site_dataset.gz
gzip -d Phosphorylation_site_dataset.gz
```

In [ ]:
reactivity_data = '/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/01_Data Analysis/01_Proteomics/02_reactivity/03_rc_analysis/reactivity_output/20251023_ratio_directionality/median_for_5+_peptides/input/02_combfiles_cysaggr_percctrl_rc.csv'
phospho_data = '/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/01_Data Analysis/01_Proteomics/05_phosphoproteomics/processed_results/expression-normalized_phosphorylation_table.xlsx'
db_dir = Path(
    "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Vinogradova Laboratory/Henry_data processing/01_data_analysis_folders/06_macrophage/macrophage/reference_dbs"
)

# read the protein sequence database used for both datasets
# we need this to check if the peptide boundaries overlap
uniprot = read_fasta(
    "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Vinogradova Laboratory/Henry_data processing/03_fasta_files/homo_sapiens/20240312/uniprotkb_proteome_UP000005640_AND_revi_2024_03_12.fasta"
)

reactivity_data = (
    pl.read_csv(reactivity_data)
    .filter(pl.col("uniprot").eq("contaminant").not_())
    .with_columns(
        pl.col("uniprot").replace(uniprot).alias("uniprot_sequence"),
        pl.concat_str(pl.col("uniprot"), pl.col("sequence"), separator="_").alias(
            "reactivity_key"
        ),
    )
    .with_columns(
        pl.col("uniprot_sequence").str.find(pl.col("sequence")).alias("peptide_start")
    )
    .with_columns(
        pl.col("peptide_start")
        .add(pl.col("sequence").str.len_chars())
        .alias("peptide_end"),
        pl.col("residue").str.replace_all(";", ","),
    )
)

reactivity_data = add_integer_location(reactivity_data)


# prepare quantified phosphorylation data
phospho_data = (
    pl.read_excel(phospho_data)
    .filter(pl.col("uniprot").eq("contaminant").not_())
    .with_columns(
        pl.col("sequence").str.replace(r"\*", "")
    )
    .with_columns(
        pl.concat_str(pl.col("uniprot"), pl.col("sequence"), separator="_").alias(
            "phospho_key"
        )
    )
)
phospho_data = (
    add_integer_location(phospho_data)
    .select("uniprot", "modification_locations", "phospho_key")
    .with_columns(pl.lit("quantified").alias("phospho_type"))
)

# prepare predicted phosphorylation data (PhosphoSite)

phosphosite_plus = (
    pl.scan_csv(
        db_dir / "Phosphorylation_site_dataset",
        separator="\t",
        truncate_ragged_lines=True,
        skip_lines=3,
        infer_schema_length=10000,
        ignore_errors=True,
    )
    .filter(pl.col("ORGANISM").eq("human"))
    .with_columns(
        pl.col("MOD_RSD")
        .str.split("-")
        .list.get(0)
        .str.slice(offset=1)
        .cast(pl.Int64)
        .alias("modification_locations"),
        pl.col("ACC_ID").alias("uniprot"),
    )
    .with_columns(
        pl.concat_str(pl.col("uniprot"), pl.col("SITE_GRP_ID"), separator="_").alias(
            "phospho_key"
        ),
        pl.lit("predicted").alias("phospho_type")
    )
    .filter(pl.col("uniprot").is_in(set(reactivity_data["uniprot"])))
    .collect()
)

phospho_data = pl.concat(
    [phospho_data, phosphosite_plus.select("uniprot", "modification_locations", "phospho_key", "phospho_type")], how = "vertical"
)

# build reactivity peptide to phospho peptide map
phospho_to_reactivity = (
    phospho_data
    .join(reactivity_data, on="uniprot", how="left")
    .filter(
        pl.col("modification_locations").is_between(
            pl.col("peptide_start"), 
            pl.col("peptide_end")
        )
    )
    .select(["phospho_key", "reactivity_key"])
    .unique(subset=["phospho_key"], keep="first")
)
phospho_to_reactivity = dict(zip(
    phospho_to_reactivity["phospho_key"],
    phospho_to_reactivity["reactivity_key"]
))

# merge using peptide to phospho map
merged = (
    phospho_data.with_columns(
        pl.col("phospho_key").replace(phospho_to_reactivity).alias("reactivity_key")
    )
    .join(other=reactivity_data, on="reactivity_key", how="inner")
    .with_columns(
        pl.col("modification_locations_right")
        .sub("modification_locations")
        .abs()
        .alias("cysteine_phosphosite_distance")
    )
)

In [ ]:
reactivity_changes = pl.read_csv(
    "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/01_Data Analysis/01_Proteomics/02_reactivity/03_rc_analysis/reactivity_output/20251023_ratio_directionality/median_for_5+_peptides/output/reactivity_changes_long_format.csv"
)
df = merged.join(
    other=reactivity_changes.with_columns(pl.concat_str(pl.col("uniprot"), pl.col("sequence"), separator="_").alias("phospho_key")).drop("uniprot"),
    on=["phospho_key"],
).unique()


df = df.select(
    "uniprot",
    #"modification_locations",
    "phospho_key",
    "phospho_type",
    "sequence",
    "peptide_start",
    "peptide_end",
    "cysteine_phosphosite_distance",
    "reactivity_change",
).unique().group_by(~cs.by_name("reactivity_change")).agg(
    pl.col("reactivity_change").any()
).group_by(
    ~cs.by_name(["modification_locations", "phospho_key", "phospho_type"])
).agg(
    pl.all().unique()
).with_columns(
    pl.when(pl.col("phospho_type").list.len() == 1)
    .then(pl.col("phospho_type").list.get(0))
    .otherwise(pl.lit("quantified"))
    .alias("phospho_type")
).with_columns(pl.col("phospho_key").list.join(","))


df.write_csv("cysteine_phosphosite.csv")

# WP vs Phospho fold change correlation scatterplot

In [ ]:
median_normalized_phospho = "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/01_Data Analysis/01_Proteomics/05_phosphoproteomics/processed_results/median-normalized_phosphorylation_table.xlsx"
expression_data = "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/01_Data Analysis/01_Proteomics/01_unenriched proteomics/03_results/20250219_3reps/04_results/volcano_plots/volcano_data.csv"

join_cols = ["uniprot", "protein", "description"]

df = pl.read_excel(median_normalized_phospho).drop(cs.contains("_d"))


df = (
    pl.read_csv(expression_data)
    .select(cs.by_name(join_cols), cs.contains("TLR4"))
    .join(other=df, on=join_cols, how="right")
    .drop(cs.by_name("ID"), cs.contains("index"))
)

phospho_lfc = pl.col("log2_FC_phospho - TLR4 vs. M0 (11689 Residues)")
wp_lfc = pl.col("log2_FC_Macrophage WP - TLR4 vs. M0 (5000 Proteins)")
cutoff = np.log2(1.5)

df = df.with_columns(
    pl.when(
        ((phospho_lfc > cutoff) & (wp_lfc > cutoff))
        | ((phospho_lfc < -cutoff) & (wp_lfc < -cutoff))
    )
    .then(pl.lit("dark_grey"))
    .when(
        (wp_lfc.abs() > cutoff)
        & (phospho_lfc > -cutoff) & (phospho_lfc < cutoff)
    )
    .then(pl.lit("yellow"))
    .when(
        (phospho_lfc.abs() > cutoff)
        & (wp_lfc > -cutoff) & (wp_lfc < cutoff)
    )
    .then(pl.lit("violet"))
    .when(
        ((phospho_lfc > cutoff) & (wp_lfc < -cutoff))
        | ((phospho_lfc < -cutoff) & (wp_lfc > cutoff))
    )
    .then(pl.lit("teal"))
    .otherwise(pl.lit("light_grey"))
    .alias("gene_color")
).write_csv('/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/01_Data Analysis/01_Proteomics/05_phosphoproteomics/processed_results/wp_phospho_scatterplot/wp_vs_phospho.csv')
